In [1]:
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
import sklearn
import imblearn
import itertools
import sys
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score
from sklearn import metrics as metrics
import warnings
warnings.filterwarnings('ignore')

In [2]:
test = pd.read_csv("UNSW_NB15_testing-set.csv")
test = test.iloc[:,:-1] 

In [3]:
train = pd.read_csv("UNSW_NB15_training-set.csv")
train = train.iloc[:,:-1] 

In [4]:
y = train["attack_cat"].values
from collections import Counter
Counter(y)

Counter({'Normal': 37000,
         'Reconnaissance': 3496,
         'Backdoor': 583,
         'DoS': 4089,
         'Exploits': 11132,
         'Analysis': 677,
         'Fuzzers': 6062,
         'Worms': 44,
         'Shellcode': 378,
         'Generic': 18871})

In [5]:
y1 = test["attack_cat"].values
from collections import Counter
Counter(y1)

Counter({'Normal': 56000,
         'Backdoor': 1746,
         'Analysis': 2000,
         'Fuzzers': 18184,
         'Shellcode': 1133,
         'Reconnaissance': 10491,
         'Exploits': 33393,
         'DoS': 12264,
         'Worms': 130,
         'Generic': 40000})

In [6]:
from sklearn.preprocessing import LabelEncoder
encodings = dict()
for c in train.columns:
    
    if train[c].dtype == "object":
        encodings[c] = LabelEncoder()
        encodings[c]
        train[c] = encodings[c].fit_transform(train[c])

In [7]:
y = train.pop("attack_cat").values
X = train.values

In [8]:
train.dtypes

id                     int64
dur                  float64
proto                  int32
service                int32
state                  int32
spkts                  int64
dpkts                  int64
sbytes                 int64
dbytes                 int64
rate                 float64
sttl                   int64
dttl                   int64
sload                float64
dload                float64
sloss                  int64
dloss                  int64
sinpkt               float64
dinpkt               float64
sjit                 float64
djit                 float64
swin                   int64
stcpb                  int64
dtcpb                  int64
dwin                   int64
tcprtt               float64
synack               float64
ackdat               float64
smean                  int64
dmean                  int64
trans_depth            int64
response_body_len      int64
ct_srv_src             int64
ct_state_ttl           int64
ct_dst_ltm             int64
ct_src_dport_l

In [9]:
from sklearn.preprocessing import StandardScaler
X = StandardScaler().fit_transform(X)

In [10]:
!pip install mlxtend

In [11]:
from mlxtend.feature_selection import SequentialFeatureSelector as sfs


In [37]:
clf = RandomForestClassifier(n_estimators=100, n_jobs=-1)

sfs1 = sfs(clf,
           k_features=20,
           forward=True,
           floating=False,
           verbose=2,
           scoring='accuracy',
           cv=5)

sfs1 = sfs1.fit(X, y)

[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    9.0s remaining:    0.0s
[Parallel(n_jobs=1)]: Done  43 out of  43 | elapsed:  2.8min finished

[2022-03-03 19:06:49] Features: 1/20 -- score: 0.734233583900536[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    5.5s remaining:    0.0s
[Parallel(n_jobs=1)]: Done  42 out of  42 | elapsed:  3.3min finished

[2022-03-03 19:10:04] Features: 2/20 -- score: 0.7812265307385269[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    5.2s remaining:    0.0s
[Parallel(n_jobs=1)]: Done  41 out of  41 | elapsed:  3.1min finished

[2022-03-03 19:13:08] Features: 3/20 -- score: 0.7955829476752532[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   

In [38]:
X_train_sfs, X_test_sfs, y_train_sfs, y_test_sfs = train_test_split(X[:, list(sfs1.k_feature_idx_)], y, test_size=0.2, random_state=42)

In [39]:
#KNN
from sklearn.neighbors import KNeighborsClassifier

neigh = KNeighborsClassifier(n_neighbors=5,weights='uniform')
neigh.fit(X_train_sfs, y_train_sfs)


KNeighborsClassifier()

In [40]:
y_pred_sfs = neigh.predict(X_test_sfs)
print(Counter(y_pred_sfs))
print(Counter(y_test_sfs))

Counter({6: 7684, 5: 3638, 3: 2315, 2: 993, 4: 977, 7: 594, 1: 138, 0: 93, 8: 35})
Counter({6: 7418, 5: 3723, 3: 2275, 4: 1212, 2: 786, 7: 723, 0: 131, 1: 117, 8: 75, 9: 7})


In [41]:
from sklearn.metrics import confusion_matrix 
from sklearn.metrics import accuracy_score 
from sklearn.metrics import classification_report 
 
results = confusion_matrix(y_test_sfs, y_pred_sfs) 
print('Confusion Matrix :')
print(results) 
print('Accuracy Score :',accuracy_score(y_test_sfs, y_pred_sfs))
print('Report : ')
print(classification_report(y_test_sfs, y_pred_sfs))

Confusion Matrix :
[[  14   19   35   48   13    0    2    0    0    0]
 [   6   18   29   41   12    1    8    2    0    0]
 [  14   21  365  293   25    5   46   15    2    0]
 [  34   42  373 1504   71   12  168   64    7    0]
 [  25   36   79  108  702    5  242   15    0    0]
 [   0    0   26   62    8 3610   12    4    1    0]
 [   0    0   32  101  114    1 7078   87    5    0]
 [   0    2   53  147   29    2  106  381    3    0]
 [   0    0    1    5    3    2   22   25   17    0]
 [   0    0    0    6    0    0    0    1    0    0]]
Accuracy Score : 0.8312989615594826
Report : 
              precision    recall  f1-score   support

           0       0.15      0.11      0.12       131
           1       0.13      0.15      0.14       117
           2       0.37      0.46      0.41       786
           3       0.65      0.66      0.66      2275
           4       0.72      0.58      0.64      1212
           5       0.99      0.97      0.98      3723
           6       0.92  

In [42]:
from sklearn.model_selection import cross_val_predict
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=10)
predicted = cross_val_predict(neigh, X_train_sfs, y_train_sfs, cv=skf)
print('Accuracy Score :',accuracy_score(y_train_sfs, predicted))
print('Report : ')
print(classification_report(y_train_sfs, predicted))

Accuracy Score : 0.830942078493889
Report : 
              precision    recall  f1-score   support

           0       0.13      0.14      0.13       546
           1       0.08      0.06      0.07       466
           2       0.40      0.43      0.41      3303
           3       0.64      0.65      0.65      8857
           4       0.70      0.60      0.65      4850
           5       0.99      0.97      0.98     15148
           6       0.92      0.96      0.94     29582
           7       0.60      0.53      0.56      2773
           8       0.50      0.20      0.29       303
           9       0.00      0.00      0.00        37

    accuracy                           0.83     65865
   macro avg       0.50      0.45      0.47     65865
weighted avg       0.83      0.83      0.83     65865



In [43]:
#SVM
from sklearn.svm import SVC

clf = SVC(gamma='auto',decision_function_shape='ovo')
clf.fit(X_train_sfs, y_train_sfs)

SVC(decision_function_shape='ovo', gamma='auto')

In [44]:
y_pred_sfs=clf.predict(X_test_sfs)
print(Counter(y_pred_sfs))
print(Counter(y_test_sfs))

Counter({6: 8353, 5: 3603, 3: 2533, 4: 1186, 7: 619, 2: 173})
Counter({6: 7418, 5: 3723, 3: 2275, 4: 1212, 2: 786, 7: 723, 0: 131, 1: 117, 8: 75, 9: 7})


In [45]:
from sklearn.metrics import confusion_matrix 
from sklearn.metrics import accuracy_score 
from sklearn.metrics import classification_report 
 
results = confusion_matrix(y_test_sfs, y_pred_sfs) 
print('Confusion Matrix :')
print(results) 
print('Accuracy Score :',accuracy_score(y_test_sfs, y_pred_sfs))
print('Report : ')
print(classification_report(y_test_sfs, y_pred_sfs))

Confusion Matrix :
[[   0    0    2   39   71    0   16    3    0    0]
 [   0    0    2   28   74    0    9    4    0    0]
 [   0    0   75  426   85    2  132   66    0    0]
 [   0    0   66 1562  192    2  379   74    0    0]
 [   0    0    4  103  673    4  381   47    0    0]
 [   0    0    2   78    9 3593   24   17    0    0]
 [   0    0   11  149   47    0 7088  123    0    0]
 [   0    0   11  143   32    2  283  252    0    0]
 [   0    0    0    0    3    0   39   33    0    0]
 [   0    0    0    5    0    0    2    0    0    0]]
Accuracy Score : 0.8042144895852311
Report : 
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       131
           1       0.00      0.00      0.00       117
           2       0.43      0.10      0.16       786
           3       0.62      0.69      0.65      2275
           4       0.57      0.56      0.56      1212
           5       1.00      0.97      0.98      3723
           6       0.85  

In [46]:
from sklearn.model_selection import cross_val_predict
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=10)
predicted = cross_val_predict(clf, X_train_sfs, y_train_sfs, cv=skf)
print('Accuracy Score :',accuracy_score(y_train_sfs, predicted))
print('Report : ')
print(classification_report(y_train_sfs, predicted))

Accuracy Score : 0.8028998709481515
Report : 
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       546
           1       0.00      0.00      0.00       466
           2       0.44      0.08      0.14      3303
           3       0.61      0.69      0.64      8857
           4       0.58      0.56      0.57      4850
           5       1.00      0.96      0.98     15148
           6       0.85      0.96      0.90     29582
           7       0.37      0.34      0.35      2773
           8       0.00      0.00      0.00       303
           9       0.00      0.00      0.00        37

    accuracy                           0.80     65865
   macro avg       0.38      0.36      0.36     65865
weighted avg       0.77      0.80      0.78     65865



In [47]:
#Logistic Regression
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(random_state=0,solver='saga',multi_class='multinomial').fit(X_train_sfs, y_train_sfs)

In [48]:
y_pred_sfs = clf.predict(X_test_sfs)
print(Counter(y_pred_sfs))
print(Counter(y_test_sfs))

Counter({6: 9659, 5: 3943, 3: 1638, 4: 607, 7: 452, 2: 168})
Counter({6: 7418, 5: 3723, 3: 2275, 4: 1212, 2: 786, 7: 723, 0: 131, 1: 117, 8: 75, 9: 7})


In [49]:
from sklearn.metrics import confusion_matrix 
from sklearn.metrics import accuracy_score 
from sklearn.metrics import classification_report 
 
print('Accuracy Score :',accuracy_score(y_test_sfs, y_pred_sfs))
print('Report : ')
print(classification_report(y_test_sfs, y_pred_sfs))

Accuracy Score : 0.7364425821339649
Report : 
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       131
           1       0.00      0.00      0.00       117
           2       0.46      0.10      0.16       786
           3       0.54      0.39      0.45      2275
           4       0.53      0.26      0.35      1212
           5       0.92      0.97      0.94      3723
           6       0.73      0.95      0.82      7418
           7       0.49      0.30      0.37       723
           8       0.00      0.00      0.00        75
           9       0.00      0.00      0.00         7

    accuracy                           0.74     16467
   macro avg       0.37      0.30      0.31     16467
weighted avg       0.69      0.74      0.70     16467



In [50]:
from sklearn.model_selection import cross_val_predict
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=10)
predicted = cross_val_predict(clf, X_train_sfs, y_train_sfs, cv=skf)
print('Accuracy Score :',accuracy_score(y_train_sfs, predicted))
print('Report : ')
print(classification_report(y_train_sfs, predicted))

Accuracy Score : 0.7392849009337281
Report : 
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       546
           1       0.00      0.00      0.00       466
           2       0.44      0.08      0.13      3303
           3       0.55      0.41      0.47      8857
           4       0.52      0.26      0.35      4850
           5       0.91      0.97      0.94     15148
           6       0.73      0.95      0.83     29582
           7       0.45      0.28      0.35      2773
           8       0.00      0.00      0.00       303
           9       0.00      0.00      0.00        37

    accuracy                           0.74     65865
   macro avg       0.36      0.29      0.31     65865
weighted avg       0.69      0.74      0.70     65865



In [51]:
#Multi Layer Perceptron
from sklearn.neural_network import MLPClassifier
clf = MLPClassifier(alpha=1)
clf.fit(X_train_sfs, y_train_sfs)
clf

MLPClassifier(alpha=1)

In [52]:
from sklearn.metrics import confusion_matrix 
from sklearn.metrics import accuracy_score 
from sklearn.metrics import classification_report 
 
results = confusion_matrix(y_test_sfs, y_pred_sfs) 
print('Confusion Matrix :')
print(results) 
print('Accuracy Score :',accuracy_score(y_test_sfs, y_pred_sfs))
print('Report : ')
print(classification_report(y_test_sfs, y_pred_sfs))

Confusion Matrix :
[[   0    0    0   16   41   38   36    0    0    0]
 [   0    0    0   13   54   31   16    3    0    0]
 [   0    0   77  169   48   68  370   54    0    0]
 [   0    0   69  883  122  107 1041   53    0    0]
 [   0    0    9   80  321   69  715   18    0    0]
 [   0    0    2   51    1 3609   43   17    0    0]
 [   0    0    0  315   13   13 7017   60    0    0]
 [   0    0   11  108    7    6  371  220    0    0]
 [   0    0    0    0    0    0   48   27    0    0]
 [   0    0    0    3    0    2    2    0    0    0]]
Accuracy Score : 0.7364425821339649
Report : 
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       131
           1       0.00      0.00      0.00       117
           2       0.46      0.10      0.16       786
           3       0.54      0.39      0.45      2275
           4       0.53      0.26      0.35      1212
           5       0.92      0.97      0.94      3723
           6       0.73  

In [53]:
from sklearn.model_selection import cross_val_predict
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=10)
predicted = cross_val_predict(clf, X_train_sfs, y_train_sfs, cv=skf)
print('Accuracy Score :',accuracy_score(y_train_sfs, predicted))
print('Report : ')
print(classification_report(y_train_sfs, predicted))

Accuracy Score : 0.7823730357549533
Report : 
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       546
           1       0.00      0.00      0.00       466
           2       0.44      0.12      0.19      3303
           3       0.58      0.63      0.60      8857
           4       0.55      0.34      0.42      4850
           5       0.99      0.96      0.98     15148
           6       0.79      0.97      0.87     29582
           7       0.45      0.27      0.34      2773
           8       0.00      0.00      0.00       303
           9       0.00      0.00      0.00        37

    accuracy                           0.78     65865
   macro avg       0.38      0.33      0.34     65865
weighted avg       0.74      0.78      0.75     65865



In [54]:
#Random Forest
from sklearn.ensemble import RandomForestRegressor
forest = RandomForestRegressor()
_ = forest.fit(X_train_sfs, y_train_sfs)
arr = forest.predict(X_train_sfs).astype(int)
print(classification_report(y_train_sfs, arr))
print('Accuracy Score :',accuracy_score(y_train_sfs, arr))

              precision    recall  f1-score   support

           0       1.00      0.08      0.15       546
           1       0.64      0.06      0.11       466
           2       0.32      0.94      0.47      3303
           3       0.86      0.47      0.61      8857
           4       0.83      0.70      0.76      4850
           5       0.70      0.97      0.81     15148
           6       0.98      0.79      0.88     29582
           7       0.89      0.68      0.77      2773
           8       0.31      0.02      0.03       303
           9       0.00      0.00      0.00        37

    accuracy                           0.77     65865
   macro avg       0.65      0.47      0.46     65865
weighted avg       0.84      0.77      0.78     65865

Accuracy Score : 0.7710468382297123


In [55]:
arr = forest.predict(X_test_sfs).astype(int)
print(classification_report(y_test_sfs, arr))
print('Accuracy Score :',accuracy_score(y_test_sfs, arr))

              precision    recall  f1-score   support

           0       1.00      0.07      0.13       131
           1       0.00      0.00      0.00       117
           2       0.27      0.83      0.40       786
           3       0.73      0.35      0.47      2275
           4       0.66      0.61      0.63      1212
           5       0.64      0.97      0.77      3723
           6       0.98      0.75      0.85      7418
           7       0.95      0.66      0.78       723
           8       0.67      0.03      0.05        75
           9       0.00      0.00      0.00         7

    accuracy                           0.72     16467
   macro avg       0.59      0.43      0.41     16467
weighted avg       0.80      0.72      0.72     16467

Accuracy Score : 0.7186494200522257


In [56]:
#NaiveBayes
from sklearn.naive_bayes import GaussianNB
classifier = GaussianNB()
classifier.fit(X_train_sfs, y_train_sfs)
arr = forest.predict(X_train_sfs).astype(int)
print(classification_report(y_train_sfs, arr))

              precision    recall  f1-score   support

           0       1.00      0.08      0.15       546
           1       0.64      0.06      0.11       466
           2       0.32      0.94      0.47      3303
           3       0.86      0.47      0.61      8857
           4       0.83      0.70      0.76      4850
           5       0.70      0.97      0.81     15148
           6       0.98      0.79      0.88     29582
           7       0.89      0.68      0.77      2773
           8       0.31      0.02      0.03       303
           9       0.00      0.00      0.00        37

    accuracy                           0.77     65865
   macro avg       0.65      0.47      0.46     65865
weighted avg       0.84      0.77      0.78     65865



In [57]:
arr = classifier.predict(X_test_sfs).astype(int)
print(classification_report(y_test_sfs, arr))
print('Accuracy Score :',accuracy_score(y_test_sfs, arr))

              precision    recall  f1-score   support

           0       0.04      0.89      0.08       131
           1       0.00      0.02      0.00       117
           2       0.11      0.01      0.01       786
           3       0.36      0.47      0.41      2275
           4       0.29      0.16      0.21      1212
           5       0.91      0.55      0.68      3723
           6       0.98      0.30      0.45      7418
           7       0.04      0.03      0.04       723
           8       0.02      0.99      0.05        75
           9       0.00      0.29      0.01         7

    accuracy                           0.35     16467
   macro avg       0.28      0.37      0.19     16467
weighted avg       0.73      0.35      0.43     16467

Accuracy Score : 0.3474221169611951


In [58]:
#DecisionTree Entropy
from sklearn.tree import DecisionTreeClassifier
clf_entropy = DecisionTreeClassifier(criterion = "entropy", random_state = 100,max_depth = 3, min_samples_leaf = 5)
  
clf_entropy.fit(X_train_sfs, y_train_sfs)
y_pred = clf_entropy.predict(X_train_sfs)
print(classification_report(y_train_sfs, y_pred))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00       546
           1       0.00      0.00      0.00       466
           2       0.00      0.00      0.00      3303
           3       0.49      0.60      0.54      8857
           4       0.00      0.00      0.00      4850
           5       1.00      0.96      0.98     15148
           6       0.69      0.95      0.80     29582
           7       0.00      0.00      0.00      2773
           8       0.00      0.00      0.00       303
           9       0.00      0.00      0.00        37

    accuracy                           0.73     65865
   macro avg       0.22      0.25      0.23     65865
weighted avg       0.61      0.73      0.66     65865



In [59]:
y_pred = clf_entropy.predict(X_test_sfs)
print(classification_report(y_test_sfs, y_pred))
print('Accuracy Score :',accuracy_score(y_test_sfs, arr))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00       131
           1       0.00      0.00      0.00       117
           2       0.00      0.00      0.00       786
           3       0.51      0.60      0.55      2275
           4       0.00      0.00      0.00      1212
           5       1.00      0.96      0.98      3723
           6       0.69      0.95      0.80      7418
           7       0.00      0.00      0.00       723
           8       0.00      0.00      0.00        75
           9       0.00      0.00      0.00         7

    accuracy                           0.73     16467
   macro avg       0.22      0.25      0.23     16467
weighted avg       0.61      0.73      0.66     16467

Accuracy Score : 0.3474221169611951


In [60]:
#DecisionTree Gini Index
clf_gini = DecisionTreeClassifier(criterion = "gini",random_state = 100,max_depth=3, min_samples_leaf=5)
clf_gini.fit(X_train_sfs, y_train_sfs)
y_pred = clf_gini.predict(X_train_sfs)
print(classification_report(y_train_sfs, y_pred))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00       546
           1       0.00      0.00      0.00       466
           2       0.00      0.00      0.00      3303
           3       0.62      0.55      0.58      8857
           4       0.00      0.00      0.00      4850
           5       1.00      0.92      0.96     15148
           6       0.67      1.00      0.80     29582
           7       0.00      0.00      0.00      2773
           8       0.00      0.00      0.00       303
           9       0.00      0.00      0.00        37

    accuracy                           0.73     65865
   macro avg       0.23      0.25      0.23     65865
weighted avg       0.61      0.73      0.66     65865



In [61]:
y_pred = clf_gini.predict(X_test_sfs)
print(classification_report(y_test_sfs, y_pred))
print('Accuracy Score :',accuracy_score(y_test_sfs, arr))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00       131
           1       0.00      0.00      0.00       117
           2       0.00      0.00      0.00       786
           3       0.66      0.56      0.60      2275
           4       0.00      0.00      0.00      1212
           5       1.00      0.92      0.96      3723
           6       0.67      1.00      0.80      7418
           7       0.00      0.00      0.00       723
           8       0.00      0.00      0.00        75
           9       0.00      0.00      0.00         7

    accuracy                           0.73     16467
   macro avg       0.23      0.25      0.24     16467
weighted avg       0.62      0.73      0.66     16467

Accuracy Score : 0.3474221169611951
